In [1]:
import page

# -*- coding: utf-8 -*-
# page.encoding='utf-8'
 
from catmap import ReactionModel

mkm_file = 'MA.mkm'
model = ReactionModel(setup_file=mkm_file)
model.output_variables += ['production_rate','rate','rate_control','coverage','selectivity_control','rxn_order']
model.run()
    
from catmap import analyze
vm = analyze.VectorMap(model)
vm.plot_variable = 'rate' #tell the model which output to plot
vm.log_scale = True #rates should be plotted on a log-scale
vm.min = 1e-25 #minimum rate to plot
vm.max = 1e3 #maximum rate to plot
vm.plot(save='rate.pdf') #draw the plot and save it as "rate.pdf"

vm.unique_only = False
vm.plot(save='all_rates.pdf')
vm.unique_only = True

vm.production_rate_map = model.production_rate_map #attach map
vm.threshold = 1e-30 #do not plot rates below this
vm.plot_variable = 'production_rate'
vm.plot(save='production_rate.pdf')

vm.descriptor_labels = ['H reactivity [eV]', 'O reactivity [eV]']
vm.subplots_adjust_kwargs = {'left':0.2,'right':0.8,'bottom':0.15}
vm.plot(save='pretty_production_rate.pdf')

vm.plot_variable = 'coverage'
vm.log_scale = False
vm.min = 0
vm.max = 1
vm.plot(save='coverage.pdf')

vm.include_labels = ['O_s']
vm.plot(save='O_coverage.pdf')

sa = analyze.ScalingAnalysis(model)
sa.plot(save='scaling.pdf')



Input line {'surface_name': 'ZrO2-111', 'site_name': '111', 'species_name': 'CH3CHOCH-H', 'formation_energy': '0.7900 ', 'bulk_structure': '[]', 'frequencies': '[]', 'other_parameters': 'Input File Tutorial.'} does not have all required fields.  Ignoring.
If you find CatMAP useful to your research, please cite both the following papers:

 Medford, A. J., Shi, C., Hoffmann, M. J., Lausche, A. C., Fitzgibbon, S. R., Bligaard, T., & Nørskov, J. K. (2015). CatMAP: a software package for descriptor-based microkinetic mapping of catalytic trends. Catalysis Letters, 145, 794-807. 
 Vijay, S., H. Heenen, H., Singh, A. R., Chan, K., & Voss, J. (2024). Number of sites-based solver for determining coverages from steady-state mean-field micro-kinetic models. Journal of Computational Chemistry, 45(9), 546-551.

header_evaluation: fail - could not save coverage_map = [[[np.float64(2.0), np.float64(2.5)], [mpf('0.0000000000000000000000000028690858238998872617747946768460792704216959830189620703832895

UnicodeEncodeError: 'gbk' codec can't encode character '\xf8' in position 598: illegal multibyte sequence

In [10]:
from glob import glob
import sys
from catmap.model import ReactionModel
#import numpy as np

model.output_variables += ['production_rate', 'rate', 'rate_control', 'coverage', 'selectivity_control', 'rxn_order']

output_variable = 'production_rate'
logfile = glob('*.log')
print(len(logfile))
#if len(logfile) > 1:
#    raise InputError('Ambiguous logfile. Ensure that only one file ends with .log')
model = ReactionModel(setup_file=logfile[0])

if output_variable == 'rate_control':
    dim = 2
else:
    dim = 1

labels = model.output_labels['production_rate']


def flatten_2d(output):
    "Helper function for flattening rate_control output"
    flat = []
    for x in output:
        flat += x
    return flat


#flatten rate_control labels
if output_variable == 'rate_control':
    flat_labels = []
    for i in labels[0]:
        for j in labels[1]:
            flat_labels.append('d' + i + '/d' + j)
    labels = flat_labels

#flatten elementary-step specific labels
if output_variable in ['rate', 'rate_constant', 'forward_rate_constant', 'reverse_rate_constant']:
    str_labels = []
    for label in labels:
        states = ['+'.join(s) for s in label]
        if len(states) == 2:
            new_label = '<->'.join(states)
        else:
            new_label = states[0] + '<->' + states[1] + '->' + states[2]
        str_labels.append(new_label)
    labels = str_labels

table = '\t'.join(list(['descriptor-' + d for d in model.descriptor_names]) + list(labels)) + '\n'

for pt, output in getattr(model, output_variable + '_map'):
    if dim == 2:
        output = flatten_2d(output)
    table += '\t'.join([str(float(i)) for i in pt + output]) + '\n'

f = open(output_variable + '_table.txt', 'w')
f.write(table)
f.close()  

1


In [11]:
from catmap.model import ReactionModel

model = ReactionModel(setup_file='MA.log')

#for MgO, cvgs in model.coverage_map:
   # print( 'descriptors:', MgO)
   #print( 'coverages', cvgs)
    
labels = model.output_labels['coverage']
for MgO ,cvg in model.coverage_map:
    print( 'descriptors',MgO)
    print( 'intermediates',labels)
    print( 'coverages', [float(c) for c in cvg])

descriptors [np.float64(2.0), np.float64(2.5)]
intermediates ('C3H5O_s', 'C3H6O_s', 'C3H6_s', 'H2O_s', 'H2_s', 'H_s', 'OH_s', 'O_s')
coverages [2.8690858238998873e-27, 7.469325251231626e-19, 5.080216762068787e-43, 9.73905338901238e-19, 4.099040629917867e-14, 1.3548743594756899e-24, 2.7890772553713893e-18, 5.1353542837294284e-26, 0.999999999999959]
descriptors [np.float64(2.0), np.float64(1.7999999999999998)]
intermediates ('C3H5O_s', 'C3H6O_s', 'C3H6_s', 'H2O_s', 'H2_s', 'H_s', 'OH_s', 'O_s')
coverages [1.6595218548998836e-25, 7.469325251231625e-19, 1.2050579640658004e-41, 1.3820821052499081e-18, 4.099040629917867e-14, 4.199917516434459e-23, 1.134888181224112e-16, 8.964772696643436e-21, 0.9999999999999589]
descriptors [np.float64(2.0), np.float64(1.0999999999999996)]
intermediates ('C3H5O_s', 'C3H6O_s', 'C3H6_s', 'H2O_s', 'H2_s', 'H_s', 'OH_s', 'O_s')
coverages [2.9269487931884883e-22, 7.46932525123054e-19, 2.8584725984770397e-40, 1.9613312191170287e-18, 4.0990406299172715e-14, 4.17931